# 00b — Pre-Annotation Otomatis Foto Ruangan (Panoptic Indoor Segmentation v5)
**Runtime:** GPU T4 (Colab Free) -> `Runtime > Change runtime type > T4 GPU`.

Notebook ini melakukan **pre-annotation otomatis** pada 40+ foto ruangan HP Anda menggunakan **GroundingDINO-tiny + SAM-vit-base**:
1. Membaca daftar resmi **74 kelas** dari `main_yolo/data.yaml` (Panoptic Indoor v5) sehingga ID & nama kelas 100% cocok.
2. Mendeteksi objek-objek ruangan umum (`chair`, `bed`, `pillow`, `blanket`, `dining table`, `laptop`, `book`, `bottle`, `cup`, `backpack`, `tv`, `shelf`, `trash-bin`, `curtain`, `door-stuff`, `cabinet-merged`, `box`, `couch`, `clock`, `light`, `mirror-stuff`, `potted plant`, `towel`, `cell phone`, `keyboard`, `mouse`, `person`, dll.).
3. Membuat mask poligon dengan SAM dan menyimpannya ke `preannotated_indoor_roboflow/` (foto + `_annotations.coco.json`) siap upload ke Roboflow untuk Anda rapikan & ambil screenshot anotasinya.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q transformers torch torchvision opencv-python-headless Pillow tqdm pyyaml

import os, glob, json, shutil, yaml
import cv2
import numpy as np
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, SamModel, SamProcessor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')


## 1. Konfigurasi Path Folder Foto Ruangan Anda
Ubah `INDOOR_PHOTO_DIR` ke folder di Google Drive yang berisi 40+ foto ruangan HP Anda.


In [ ]:
PROJECT_DIR = '/content/drive/MyDrive/cpmk3-instance-segmentation'

# === GANTI PATH INI KE FOLDER 40+ FOTO RUANGAN HP ANDA ===
INDOOR_PHOTO_DIR = '/content/drive/MyDrive/FOLDER_FOTO_RUANGAN_ANDA'

OUTPUT_DIR  = os.path.join(PROJECT_DIR, 'preannotated_indoor_roboflow')
PREVIEW_DIR = os.path.join(PROJECT_DIR, 'preannotated_indoor_preview')
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PREVIEW_DIR, exist_ok=True)

# Baca 74 kelas resmi dari Panoptic Indoor v5
with open(os.path.join(PROJECT_DIR, 'datasets', 'main_yolo', 'data.yaml')) as f:
    V5_CLASSES = yaml.safe_load(f)['names']
print(f'Loaded {len(V5_CLASSES)} kelas resmi Panoptic Indoor v5.')

# Cari semua foto di INDOOR_PHOTO_DIR (termasuk subfolder jika ada)
IMG_EXTS = ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG')
photo_paths = []
for ext in IMG_EXTS:
    photo_paths.extend(glob.glob(os.path.join(INDOOR_PHOTO_DIR, '**', ext), recursive=True))
photo_paths = sorted(set(photo_paths))
print(f'Total foto ruangan ditemukan: {len(photo_paths)}')
if len(photo_paths) == 0:
    print('Isi MyDrive:', os.listdir('/content/drive/MyDrive'))


## 2. Muat Model GroundingDINO-tiny + SAM-vit-base


In [ ]:
DINO_ID = 'IDEA-Research/grounding-dino-tiny'
SAM_ID  = 'facebook/sam-vit-base'

print('Memuat GroundingDINO...')
dino_processor = AutoProcessor.from_pretrained(DINO_ID)
dino_model = AutoModelForZeroShotObjectDetection.from_pretrained(DINO_ID).to(device).eval()

print('Memuat SAM (Segment Anything)...')
sam_processor = SamProcessor.from_pretrained(SAM_ID)
sam_model = SamModel.from_pretrained(SAM_ID).to(device).eval()
print('Model siap!')


## 3. Jalankan Pre-Annotation Objek Ruangan -> COCO Segmentation JSON
Objek ruangan yang dideteksi otomatis dipetakan ke nama kelas persis milik **Panoptic Indoor v5**.


In [ ]:
# Pemetaan prompt deteksi -> nama kelas persis di Panoptic Indoor v5
PROMPT_TO_V5 = {
    'chair': 'chair',
    'bed': 'bed',
    'pillow': 'pillow',
    'blanket': 'blanket',
    'table': 'dining table',
    'desk': 'dining table',
    'couch': 'couch',
    'sofa': 'couch',
    'cabinet': 'cabinet-merged',
    'wardrobe': 'cabinet-merged',
    'shelf': 'shelf',
    'door': 'door-stuff',
    'curtain': 'curtain',
    'window': 'window-other',
    'mirror': 'mirror-stuff',
    'laptop': 'laptop',
    'keyboard': 'keyboard',
    'mouse': 'mouse',
    'tv': 'tv',
    'monitor': 'tv',
    'bottle': 'bottle',
    'cup': 'cup',
    'mug': 'cup',
    'book': 'book',
    'backpack': 'backpack',
    'bag': 'handbag',
    'box': 'box',
    'trash bin': 'trash-bin',
    'clock': 'clock',
    'lamp': 'light',
    'potted plant': 'potted plant',
    'towel': 'towel',
    'phone': 'cell phone',
    'person': 'person',
}

# Bagi prompt ke 2 grup agar GroundingDINO tetap fokus & akurat
PROMPT_GROUPS = [
    'chair. bed. pillow. blanket. table. desk. couch. cabinet. wardrobe. shelf. door. curtain. window. mirror.',
    'laptop. keyboard. mouse. tv. monitor. bottle. cup. book. backpack. bag. box. trash bin. clock. lamp. potted plant. towel. phone. person.',
]

BOX_THRESHOLD = 0.30        # Naikkan ke 0.35 jika terlalu banyak deteksi, turunkan ke 0.25 jika kurang
MAX_OBJECTS_PER_IMAGE = 10  # Maksimal objek per foto ruangan
POLY_EPSILON_RATIO = 0.004

class_to_cat_id = {name: i + 1 for i, name in enumerate(V5_CLASSES)}

coco_output = {
    'info': {'description': 'Pre-annotated Indoor HP Test Set (Panoptic Indoor v5 classes)'},
    'licenses': [],
    'categories': [
        {'id': cid, 'name': cname, 'supercategory': 'indoor'}
        for cname, cid in class_to_cat_id.items()
    ],
    'images': [],
    'annotations': [],
}


def map_label_to_v5(raw_label):
    lbl = raw_label.strip().lower()
    if lbl in PROMPT_TO_V5 and PROMPT_TO_V5[lbl] in class_to_cat_id:
        return PROMPT_TO_V5[lbl]
    for k, v in PROMPT_TO_V5.items():
        if k in lbl and v in class_to_cat_id:
            return v
    return None


def mask_to_polygons(bin_mask, epsilon_ratio=0.004, min_area=150):
    contours, _ = cv2.findContours(bin_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    polygons = []
    for cnt in contours:
        if cv2.contourArea(cnt) < min_area:
            continue
        eps = epsilon_ratio * cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, eps, True)
        if len(approx) >= 3:
            poly = approx.flatten().tolist()
            if len(poly) >= 6:
                polygons.append([float(x) for x in poly])
    return polygons


def nms_boxes(dets, iou_thr=0.6):
    # dets: list of (score, box, v5_class)
    dets = sorted(dets, key=lambda x: -x[0])
    kept = []
    for s, b, c in dets:
        overlap = False
        for _, kb, _ in kept:
            ix1, iy1 = max(b[0], kb[0]), max(b[1], kb[1])
            ix2, iy2 = min(b[2], kb[2]), min(b[3], kb[3])
            inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
            area_a = (b[2] - b[0]) * (b[3] - b[1])
            area_b = (kb[2] - kb[0]) * (kb[3] - kb[1])
            iou = inter / max(area_a + area_b - inter, 1e-6)
            if iou > iou_thr:
                overlap = True
                break
        if not overlap:
            kept.append((s, b, c))
    return kept


ann_id = 1
rng = np.random.default_rng(42)
PALETTE = [tuple(int(c) for c in rng.integers(50, 255, size=3)) for _ in range(len(V5_CLASSES) + 1)]

for img_id, img_path in enumerate(tqdm(photo_paths, desc='Pre-annotating indoor'), start=1):
    out_fname = f'indoor_{img_id:03d}_{os.path.basename(img_path)}'
    shutil.copy2(img_path, os.path.join(OUTPUT_DIR, out_fname))

    pil_img = Image.open(img_path).convert('RGB')
    w, h = pil_img.size

    coco_output['images'].append({
        'id': img_id,
        'file_name': out_fname,
        'width': w,
        'height': h,
    })

    # 1. Deteksi dengan GroundingDINO (2 grup prompt)
    raw_dets = []
    for prompt in PROMPT_GROUPS:
        inputs = dino_processor(images=pil_img, text=prompt, return_tensors='pt').to(device)
        with torch.no_grad():
            outputs = dino_model(**inputs)
        res = dino_processor.post_process_grounded_object_detection(
            outputs, inputs.input_ids,
            box_threshold=BOX_THRESHOLD, text_threshold=0.25,
            target_sizes=[(h, w)],
        )[0]
        for sc, bx, lb in zip(res['scores'].cpu().numpy(), res['boxes'].cpu().numpy(), res['labels']):
            v5_cls = map_label_to_v5(lb)
            if v5_cls is not None:
                raw_dets.append((float(sc), bx, v5_cls))

    dets = nms_boxes(raw_dets, iou_thr=0.6)[:MAX_OBJECTS_PER_IMAGE]

    # 2. Segmentasi tiap box dengan SAM
    vis_img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    overlay = vis_img.copy()

    for sc, box, v5_cls in dets:
        cat_id = class_to_cat_id[v5_cls]
        input_boxes = [[[float(box[0]), float(box[1]), float(box[2]), float(box[3])]]]
        sam_inputs = sam_processor(pil_img, input_boxes=input_boxes, return_tensors='pt').to(device)
        with torch.no_grad():
            sam_outputs = sam_model(**sam_inputs)

        masks = sam_processor.image_processor.post_process_masks(
            sam_outputs.pred_masks.cpu(),
            sam_inputs['original_sizes'].cpu(),
            sam_inputs['reshaped_input_sizes'].cpu(),
        )[0]
        best_idx = int(np.argmax(sam_outputs.iou_scores.cpu().numpy()[0, 0]))
        bin_mask = masks[0, best_idx].numpy().astype(np.uint8)

        polygons = mask_to_polygons(bin_mask, epsilon_ratio=POLY_EPSILON_RATIO)
        if not polygons:
            continue

        ys, xs = np.where(bin_mask == 1)
        x1, y1, x2, y2 = float(xs.min()), float(ys.min()), float(xs.max()), float(ys.max())
        coco_output['annotations'].append({
            'id': ann_id,
            'image_id': img_id,
            'category_id': cat_id,
            'segmentation': polygons,
            'area': float(bin_mask.sum()),
            'bbox': [x1, y1, x2 - x1, y2 - y1],
            'iscrowd': 0,
        })
        ann_id += 1

        color = PALETTE[cat_id % len(PALETTE)]
        overlay[bin_mask == 1] = color
        cv2.rectangle(vis_img, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
        cv2.putText(vis_img, f'{v5_cls} {sc:.2f}', (int(x1), max(25, int(y1) - 6)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

    blended = cv2.addWeighted(overlay, 0.35, vis_img, 0.65, 0)
    cv2.imwrite(os.path.join(PREVIEW_DIR, out_fname), blended)

coco_json_path = os.path.join(OUTPUT_DIR, '_annotations.coco.json')
with open(coco_json_path, 'w', encoding='utf-8') as f:
    json.dump(coco_output, f, indent=2)

print(f'\nSELESAI!')
print(f'Total foto ruangan : {len(coco_output["images"])}')
print(f'Total anotasi objek: {len(coco_output["annotations"])}')
print(f'Folder Roboflow    : {OUTPUT_DIR}')
print(f'Folder Preview     : {PREVIEW_DIR}')
